In [21]:
import os 
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F 
import time
import torchvision.models as models
from matplotlib import pyplot as plt 
import optuna

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

### Load Data 

In [6]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2),   # trial and error to find these values for params
    transforms.Resize((224,224)), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
dataset_path = "./dataset/"
dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms) # transform : it will help if the images are smaller
len(dataset)

2300

In [10]:
dataset.classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [12]:
num_classes = len(dataset.classes)
num_classes

6

In [49]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size

(1725, 575)

In [51]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [53]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

### Model Training & Hyperparameter Tuning

In [56]:
# Load the pre-trained ResNet model
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.model = models.resnet50(weights='DEFAULT')
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False
            
        # Unfreeze layer4 and fc layers
        for param in self.model.layer4.parameters():
            param.requires_grad = True            
            
        # Replace the final fully connected layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [58]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    
    # Load the model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)
    
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    # Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        
        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)
        
        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")
    
    return accuracy

In [60]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-11-18 10:58:58,088] A new study created in memory with name: no-name-32ab2a9e-bbca-4c73-bffa-d46f3611a6cb
[I 2026-11-18 11:14:42,612] Trial 0 finished with value: 75.47826086956522 and parameters: {'lr': 0.0010611236976536104, 'dropout_rate': 0.4868896383612721}. Best is trial 0 with value: 75.47826086956522.


Execution time: 944.0787868499756 seconds


[I 2026-11-18 11:31:47,077] Trial 1 finished with value: 69.21739130434783 and parameters: {'lr': 2.9802087594403716e-05, 'dropout_rate': 0.2795330582512179}. Best is trial 0 with value: 75.47826086956522.


Execution time: 1024.0059261322021 seconds


[I 2026-11-18 11:48:33,258] Trial 2 finished with value: 67.47826086956522 and parameters: {'lr': 2.8760778170418452e-05, 'dropout_rate': 0.5121072494873481}. Best is trial 0 with value: 75.47826086956522.


Execution time: 1005.7106890678406 seconds


... (15 outputs left out)

Execution time: 1129.77232503891 seconds


[I 2026-11-18 14:41:43,594] Trial 13 pruned. 
[I 2026-11-18 15:00:00,432] Trial 14 finished with value: 80.34782608695652 and parameters: {'lr': 0.00034777471033894545, 'dropout_rate': 0.22395356171442568}. Best is trial 8 with value: 81.21739130434783.


Execution time: 1096.2599680423737 seconds


[I 2026-11-18 15:06:13,652] Trial 15 pruned. 
[I 2026-11-18 15:12:24,941] Trial 16 pruned. 
[I 2026-11-18 15:18:44,364] Trial 17 pruned. 
[I 2026-11-18 15:25:00,308] Trial 18 pruned. 
[I 2026-11-18 15:31:26,634] Trial 19 pruned. 


In [62]:
study.best_params

{'lr': 0.0005726571567981019, 'dropout_rate': 0.2121012010156194}